# 6. Hyperparameteroptimierung der klassischen Modellen mit Optuna

In diesem Notebook werden die aussichtsreichsten klassischen Modelle aus der Baseline-Phase mit Optuna optimiert. Die Optimierung erfolgt ausschließlich auf dem Trainingsdatensatz. Der Testdatensatz bleibt weiterhin unberührt und wird erst in der finalen Evaluation verwendet.

## 6.1. Ziel der Optimierung

Die bisherigen Experimente zeigen, dass das Szenario `Text + categorical` eine gute Balance zwischen Modellleistung und reduzierter Merkmalskomplexität liefert. Daher wird dieses Szenario für die Hyperparameteroptimierung als feste Datengrundlage verwendet.

Die Reihenfolge der Optimierung orientiert sich an der bisherigen Laufzeit: Zuerst wird `SGDClassifier(loss="hinge")` optimiert, da dieses Modell sehr schnell trainiert und dadurch mehr Suchläufe erlaubt. Anschließend wird `Logistic Regression balanced` optimiert, da dieses Modell in der Baseline den höchsten Macro-F1-Score erzielt hat.

Als Hauptmetrik wird weiterhin `Macro-F1` verwendet. Zusätzlich werden Standardabweichung, Train-Test-Gap, Weighted-F1, Balanced Accuracy und Laufzeit dokumentiert, damit die Modellauswahl nicht nur auf einem einzelnen Mittelwert basiert.

In [2]:
from pathlib import Path
import os
import re
import tempfile
import sys

import mlflow
import numpy as np
import optuna
import pandas as pd

from IPython.display import display
from joblib import Memory, dump
from tqdm.autonotebook import tqdm

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import f1_score, make_scorer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

d:\toydev\schwarz-test\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 6.2. Daten laden

Es werden nur die Trainingsdaten geladen. Die Hyperparameteroptimierung darf den Testdatensatz nicht verwenden, damit die finale Modellbewertung unverzerrt bleibt.

In [3]:

# Hugging-Face-Konfiguration: Der Token wird nicht im Notebook gespeichert.
# Falls HF_TOKEN bereits als Umgebungsvariable oder Notebook-Variable existiert,
# wird er fuer Downloads vom Hugging Face Hub verwendet.
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
hf_token = globals().get("HF_TOKEN", os.environ.get("HF_TOKEN"))
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
else:
    hf_token = None

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import importlib
import model_evaluation
import model_evaluation.utils as model_evaluation_utils

importlib.reload(model_evaluation_utils)
importlib.reload(model_evaluation)

from model_evaluation import (
    combine_columns_as_text,
    make_fasttext_label,
    safe_model_filename,
    ensure_val_metric_aliases,
    select_validation_result_columns,
)


sklearn_memory = Memory(
    location=project_root / "cache" / "sklearn",
    verbose=0,
)

model_output_dir = project_root / "models" / "optuna"
model_output_dir.mkdir(parents=True, exist_ok=True)


def save_fitted_model(model, model_name, X, y):
    """Fitten auf allen Trainingsdaten und als Joblib-Datei speichern."""
    model_path = model_output_dir / f"{safe_model_filename(model_name)}.joblib"
    model.fit(X, y)
    dump(model, model_path)
    return model_path

train_df = pd.read_csv(
    project_root / "data" / "processed" / "train_data.csv",
    sep=";",
    index_col=0,
    encoding="utf-8",
)

print("Trainingsdaten:", train_df.shape)
display(train_df.head())

Trainingsdaten: (45920, 28)


,name,geber,art,jahr,anschrift,politikbereich,zweck,betrag,empfaengerid,name_normalized,...,anschrift_standardised,zweck_standardised,empfaengerid_standardised,name_standardised_before_modelling,geber_standardised_before_modelling,art_standardised_before_modelling,anschrift_standardised_before_modelling,zweck_standardised_before_modelling,empfaengerid_standardised_before_modelling,betrag_standardised
id,,,,,,,,,,,,,,,,,,,,,
33655,Cashmere Radio e. V.,Senatsverwaltung für Kultur und Gesellschaftli...,Projektförderung,2024,"Frankfurter Allee 7, 10247 Berlin",Kultur,"signal2noise ? Art, Aesthetics And Social Prac...",120000,vr_035983,cashmere radio e. v.,...,"frankfurter allee 7, 10247 berlin-bezirk fried...","signal2noise ? art, aesthetics and social prac...",vr_035983,cashmere radio e. v.,senatsverwaltung für kultur und gesellschaftli...,projektförderung,"Frankfurter Allee 7, 10247 Berlin-Bezirk Fried...","signal2noise ? art, aesthetics and social prac...",vr_035983,11.695255
105910,Merantix Labs GmbH,"Senatsverwaltung für Wirtschaft, Energie und B...",Projektförderung,2022,"Max-Urich-Straße 3, 13355 Berlin",Wirtschaft,Errichtung einer Betriebsstätte,5481380,hrb_221397,merantix labs gmbh,...,"c/o ai campus, fachbereich bionik und evolutio...",errichtung einer betriebsstätte,hrb_221397,merantix labs gmbh,"senatsverwaltung für wirtschaft, energie und b...",projektförderung,"c/o AI Campus, Fachbereich Bionik und Evolutio...",errichtung einer betriebsstätte,hrb_221397,15.516868
67465,Georg-Kolbe-Stiftung,Senatsverwaltung für Kultur und Europa,Projektförderung,2020,"Sensburger Allee 25, 14055 Berlin",Kultur,Der absolute Tanz- Festival sculpture,70000,spr_100011,georg-kolbe-stiftung,...,"sensburger allee 25, 14055 berlin-bezirk charl...",der absolute tanz- festival sculpture,spr_100011,georg kolbe-stiftung,senatsverwaltung für kultur und europa,projektförderung,"Sensburger Allee 25, 14055 Berlin-Bezirk Charl...",der absolute tanz- festival sculpture,spr_100011,11.156265
159301,Verschiedene Gesellschaften bürgerlichen Rechts,"Senatsverwaltung für Inneres, Digitalisierung ...",Projektförderung,2023,'---',Sport,Kosten für die Beschäftigung von Übungsleitern,1380,NaN,verschiedene gesellschaften bürgerlichen rechts,...,deutschland,kosten für die beschäftigung von übungsleitern,NaN,verschiedene gesellschaften bürgerlichen rechts,"senatsverwaltung für inneres, digitalisierung ...",projektförderung,Deutschland,kosten für die beschäftigung von übungsleitern,NaN,7.230563
19495,Berliner Rugby-Club,Senatsverwaltung für Inneres und Sport,Projektförderung,2020,"Scharfestraße 12, 14169 Berlin",Sport,anteilige Finanzierung des Spielbetriebes der ...,9000,vr_002548,berliner rugby-club,...,"scharfestrasse 12, 14169 berlin-bezirk steglit...",anteilige finanzierung des spielbetriebes der ...,vr_002548,berliner rugby-club,senatsverwaltung für inneres und sport,projektförderung,"Scharfestraße 12, 14169 Berlin-Bezirk Steglitz...",anteilige finanzierung des spielbetriebes der ...,vr_002548,9.105091


## 6.3. Optimierungsszenario festlegen

Für das Tuning wird das Szenario `Text + categorical` verwendet. Numerische Merkmale werden in diesem Szenario bewusst nicht verwendet, da die Ablation darauf hindeutet, dass die Text- und kategorialen Merkmale für diese Aufgabe ausreichend stark sind.

In [8]:
target_column = "politikbereich"

text_features_optuna = [
    "name_standardised",
    "geber_standardised",
    "anschrift_standardised",
    "zweck_standardised",
]

categorical_features_optuna = [
    "art_standardised",
    "jahr",
]

numeric_features_optuna = []

feature_columns_optuna = (
    text_features_optuna
    + categorical_features_optuna
    + numeric_features_optuna
)

X_train = train_df[feature_columns_optuna].copy()
y_train = train_df[target_column].copy()

print("Verwendete Eingabespalten:")
display(pd.DataFrame({"Trainingsspalte": feature_columns_optuna}))

print("Anzahl Klassen:", y_train.nunique())

Verwendete Eingabespalten:


,Trainingsspalte
0,name_standardised
1,geber_standardised
2,anschrift_standardised
3,zweck_standardised
4,art_standardised
5,jahr


Anzahl Klassen: 32


## 6.4. Vorverarbeitung und Evaluationsrahmen

Die Vorverarbeitung entspricht der Baseline-Logik: Jede Textspalte wird separat mit TF-IDF verarbeitet, kategoriale Merkmale werden One-Hot-codiert. Für die Bewertung wird Stratified K-Fold Cross-Validation verwendet, damit die Klassenverteilung in den Folds möglichst stabil bleibt.

In [9]:
def flatten_column(values):
    """Konvertiert eine einzelne Spalte in ein eindimensionales String-Array."""
    return np.asarray(values, dtype=object).ravel()


text_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="",
            ),
        ),
        (
            "flatten",
            FunctionTransformer(
                flatten_column,
                validate=False,
            ),
        ),
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                min_df=2,
                max_df=0.90,
                ngram_range=(1, 2),
                sublinear_tf=True,
                max_features=150_000,
                dtype=np.float32,
            ),
        ),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore"),
        ),
    ]
)


def build_preprocessor(
    text_features,
    categorical_features,
):
    transformers = [
        (
            f"tfidf_{column}",
            text_transformer,
            [column],
        )
        for column in text_features
    ]

    if categorical_features:
        transformers.append(
            (
                "categorical",
                categorical_transformer,
                categorical_features,
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )


preprocessor_optuna = build_preprocessor(
    text_features_optuna,
    categorical_features_optuna,
)

n_splits = 4
cv = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=42,
)

scoring = {
    "macro_f1": "f1_macro",
    "weighted_f1": "f1_weighted",
    "balanced_accuracy": "balanced_accuracy",
    "accuracy": "accuracy",
}

main_metric = "macro_f1"

## 6.5. Hilfsfunktionen für Optuna

Für jeden Trial wird eine Cross-Validation durchgeführt. Optuna maximiert den durchschnittlichen Test-Macro-F1-Score. Zusätzlich werden Train-Macro-F1, Generalization Gap, Standardabweichung und Laufzeit gespeichert, damit Overfitting und Stabilität beurteilt werden können.

In [10]:
def summarize_cv_results(cv_results):
    summary = {
        "fit_time_total_seconds": round(cv_results["fit_time"].sum(), 2),
        "fit_time_mean_seconds": round(cv_results["fit_time"].mean(), 2),
        "score_time_mean_seconds": round(cv_results["score_time"].mean(), 2),
    }

    for metric_name in scoring.keys():
        validation_scores = cv_results[f"test_{metric_name}"]
        summary[f"val_{metric_name}_mean"] = round(validation_scores.mean(), 4)
        summary[f"val_{metric_name}_std"] = round(validation_scores.std(), 4)
        summary[f"{metric_name}_mean"] = summary[f"val_{metric_name}_mean"]
        summary[f"{metric_name}_std"] = summary[f"val_{metric_name}_std"]

        train_key = f"train_{metric_name}"

        if train_key in cv_results:
            train_scores = cv_results[train_key]
            summary[f"train_{metric_name}_mean"] = round(train_scores.mean(), 4)
            summary[f"train_{metric_name}_std"] = round(train_scores.std(), 4)
            summary[f"generalization_gap_{metric_name}"] = round(
                summary[f"train_{metric_name}_mean"]
                - summary[f"val_{metric_name}_mean"],
                4,
            )

    return summary


def evaluate_model(model):
    cv_results = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=True,
    )

    return summarize_cv_results(cv_results)


def log_optuna_trial(trial, model_name, summary):
    trial.set_user_attr("model_name", model_name)

    for metric_name, metric_value in summary.items():
        if pd.notna(metric_value):
            trial.set_user_attr(metric_name, metric_value)


def build_result_table(study, model_name):
    rows = []

    for trial in study.trials:
        if trial.value is None:
            continue

        row = {
            "Modell": model_name,
            "trial_number": trial.number,
            "objective_value": trial.value,
        }
        row.update(trial.params)
        row.update(trial.user_attrs)
        rows.append(row)

    return (
        pd.DataFrame(rows)
        .sort_values("objective_value", ascending=False)
        .reset_index(drop=True)
    )

## 6.6. Optuna und MLflow vorbereiten

Die Optuna-Studien werden in einer SQLite-Datei gespeichert. Dadurch können bereits berechnete Trials später wiederverwendet werden. Zusätzlich werden die besten Ergebnisse als CSV-Dateien gespeichert und in MLflow dokumentiert.

In [11]:
optuna_storage_path = (
    project_root
    / "data"
    / "processed"
    / "optuna_studies.db"
)

optuna_storage_url = "sqlite:///" + optuna_storage_path.as_posix()

optuna_results_dir = project_root / "data" / "processed"
optuna_results_dir.mkdir(parents=True, exist_ok=True)

mlflow_tracking_uri = (project_root / "mlruns").as_uri()
mlflow.set_tracking_uri(mlflow_tracking_uri)
mlflow.set_experiment("politikbereich_classifier_optuna")

sampler = optuna.samplers.TPESampler(seed=42)

print("Optuna storage:", optuna_storage_url)
print("MLflow tracking URI:", mlflow_tracking_uri)

Optuna storage: sqlite:///d:/toydev/schwarz-test/data/processed/optuna_studies.db
MLflow tracking URI: file:///d:/toydev/schwarz-test/mlruns


## 6.7. Studie 1: SGDClassifier mit Hinge Loss

Zuerst wird das schnelle SGD-SVM-Modell optimiert. Der Schwerpunkt liegt auf der Regularisierung (`alpha`), der Penalty-Struktur und der Konvergenztoleranz. `class_weight="balanced"` bleibt gesetzt, weil die Zielvariable deutlich unausgewogen ist.

In [7]:
def objective_sgd_hinge(trial):
    penalty = trial.suggest_categorical(
        "penalty",
        [
            "l2",
            "elasticnet",
        ],
    )

    classifier_params = {
        "loss": "hinge",
        "penalty": penalty,
        "alpha": trial.suggest_float(
            "alpha",
            1e-6,
            1e-3,
            log=True,
        ),
        "class_weight": "balanced",
        "max_iter": trial.suggest_categorical(
            "max_iter",
            [
                1000,
                2000,
                5000,
            ],
        ),
        "tol": trial.suggest_categorical(
            "tol",
            [
                1e-4,
                1e-3,
                1e-2,
            ],
        ),
        "average": trial.suggest_categorical(
            "average",
            [
                False,
                True,
            ],
        ),
        "random_state": 42,
        "n_jobs": -1,
    }

    if penalty == "elasticnet":
        classifier_params["l1_ratio"] = trial.suggest_float(
            "l1_ratio",
            0.05,
            0.50,
        )

    model = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor_optuna,
            ),
            (
                "classifier",
                SGDClassifier(**classifier_params),
            ),
        ],
        memory=sklearn_memory,
    )

    summary = evaluate_model(model)
    log_optuna_trial(
        trial,
        "SGD hinge",
        summary,
    )

    return summary[f"val_{main_metric}_mean"]


sgd_hinge_study = optuna.create_study(
    study_name="sgd_hinge_text_categorical_macro_f1_cv4",
    direction="maximize",
    storage=optuna_storage_url,
    load_if_exists=True,
    sampler=sampler,
)

n_trials_sgd_hinge = 50

sgd_hinge_study.optimize(
    objective_sgd_hinge,
    n_trials=n_trials_sgd_hinge,
    show_progress_bar=True,
)

sgd_hinge_results = build_result_table(
    sgd_hinge_study,
    "SGD hinge",
)

display(
    sgd_hinge_results[
        select_validation_result_columns(
            sgd_hinge_results,
            extra_columns=[
                "Modell",
                "trial_number",
                "objective_value",
                "penalty",
                "alpha",
                "l1_ratio",
                "C",
                "class_weight",
                "max_iter",
                "tol",
                "average",
            ],
        )
    ].head(10)
)

[I 2026-07-27 20:54:54,976] A new study created in RDB with name: sgd_hinge_text_categorical_macro_f1
Best trial: 0. Best value: 0.6747:   2%|▏         | 1/50 [00:35<28:53, 35.38s/it]

[I 2026-07-27 20:55:30,355] Trial 0 finished with value: 0.6747 and parameters: {'penalty': 'elasticnet', 'alpha': 0.000157029708840554, 'max_iter': 1000, 'tol': 0.001, 'average': False, 'l1_ratio': 0.48645943347289744}. Best is trial 0 with value: 0.6747.


Best trial: 1. Best value: 0.8053:   4%|▍         | 2/50 [00:59<23:05, 28.86s/it]

[I 2026-07-27 20:55:54,644] Trial 1 finished with value: 0.8053 and parameters: {'penalty': 'l2', 'alpha': 3.5113563139704077e-06, 'max_iter': 5000, 'tol': 0.01, 'average': True}. Best is trial 1 with value: 0.8053.


Best trial: 1. Best value: 0.8053:   6%|▌         | 3/50 [01:25<21:21, 27.26s/it]

...

Best trial: 16. Best value: 0.8509:  98%|█████████▊| 49/50 [25:52<00:29, 29.47s/it]

[I 2026-07-27 21:20:47,238] Trial 48 finished with value: 0.8226 and parameters: {'penalty': 'l2', 'alpha': 1.6228609790975253e-05, 'max_iter': 5000, 'tol': 0.01, 'average': False}. Best is trial 16 with value: 0.8509.


Best trial: 16. Best value: 0.8509: 100%|██████████| 50/50 [26:30<00:00, 31.81s/it]

[I 2026-07-27 21:21:25,318] Trial 49 finished with value: 0.8326 and parameters: {'penalty': 'elasticnet', 'alpha': 3.1569492884255573e-06, 'max_iter': 5000, 'tol': 0.0001, 'average': False, 'l1_ratio': 0.19567752043574008}. Best is trial 16 with value: 0.8509.


,Modell,trial_number,objective_value,penalty,alpha,max_iter,tol,average,l1_ratio,accuracy_mean,...,train_accuracy_mean,train_accuracy_std,train_balanced_accuracy_mean,train_balanced_accuracy_std,train_macro_f1_mean,train_macro_f1_std,train_weighted_f1_mean,train_weighted_f1_std,weighted_f1_mean,weighted_f1_std
0,SGD hinge,16,0.8509,elasticnet,0.000008,2000,0.0001,False,0.066307,0.9410,...,0.9930,0.0003,0.9908,0.0023,0.9693,0.0069,0.9931,0.0003,0.9412,0.0023
1,SGD hinge,42,0.8478,elasticnet,0.000002,5000,0.0001,False,0.115795,0.9389,...,0.9943,0.0005,0.9926,0.0019,0.9740,0.0063,0.9944,0.0005,0.9389,0.0008
2,SGD hinge,41,0.8451,elasticnet,0.000002,5000,0.0001,False,0.076574,0.9375,...,0.9942,0.0006,0.9920,0.0021,0.9759,0.0043,0.9943,0.0006,0.9374,0.0013
3,SGD hinge,33,0.8450,elasticnet,0.000004,1000,0.0001,False,0.087296,0.9387,...,0.9941,0.0005,0.9913,0.0022,0.9731,0.0062,0.9941,0.0005,0.9387,0.0017
4,SGD hinge,22,0.8440,elasticnet,0.000004,2000,0.0001,False,0.107390,0.9395,...,0.9938,0.0007,0.9929,0.0017,0.9711,0.0089,0.9940,0.0007,0.9395,0.0022
5,SGD hinge,30,0.8428,elasticnet,0.000001,1000,0.0001,False,0.090399,0.9361,...,0.9944,0.0006,0.9893,0.0042,0.9712,0.0099,0.9945,0.0005,0.9361,0.0021
6,SGD hinge,39,0.8414,elasticnet,0.000002,5000,0.0001,False,0.074898,0.9378,...,0.9944,0.0010,0.9905,0.0027,0.9712,0.0069,0.9946,0.0009,0.9377,0.0014
7,SGD hinge,11,0.8412,elasticnet,0.000009,2000,0.0001,False,0.297018,0.9391,...,0.9908,0.0003,0.9909,0.0015,0.9686,0.0063,0.9908,0.0003,0.9392,0.0026
8,SGD hinge,43,0.8409,elasticnet,0.000003,5000,0.0001,False,0.110676,0.9371,...,0.9942,0.0004,0.9928,0.0019,0.9726,0.0081,0.9943,0.0004,0.9374,0.0021
9,SGD hinge,24,0.8408,elasticnet,0.000002,2000,0.0001,False,0.119665,0.9370,...,0.9941,0.0012,0.9929,0.0027,0.9710,0.0103,0.9944,0.0008,0.9373,0.0020


## 6.8. Studie 2: Logistische Regression (Aufgrund der Zeitaufwand noch nicht ausgeführt)

Anschließend wird die reguläre logistische Regression optimiert. Dieses Modell war in der Baseline sehr stark, ist aber rechenintensiver als das SGD-Modell. Deshalb wird es nach dem schnellen SGD-Modell optimiert.

In [ ]:
def objective_logistic_regression(trial):
    classifier_params = {
        "C": trial.suggest_float(
            "C",
            0.5,
            2.0,
            log=True,
        ),
        "penalty": "l2",
        "solver": "saga",
        "class_weight": "balanced",
        "max_iter": trial.suggest_categorical(
            "max_iter",
            [
                800,
                1000,
            ],
        ),
        "tol": trial.suggest_categorical(
            "tol",
            [
                5e-3,
                1e-2,
            ],
        ),
        "n_jobs": -1,
        "random_state": 42,
    }

    model = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor_optuna,
            ),
            (
                "classifier",
                LogisticRegression(**classifier_params),
            ),
        ],
        memory=sklearn_memory,
    )

    summary = evaluate_model(model)
    log_optuna_trial(
        trial,
        "Logistic Regression",
        summary,
    )

    return summary[f"val_{main_metric}_mean"]


logreg_study = optuna.create_study(
    study_name="logreg_text_categorical_macro_f1_cv4",
    direction="maximize",
    storage=optuna_storage_url,
    load_if_exists=True,
    sampler=sampler,
)

n_trials_logreg = 30

logreg_study.optimize(
    objective_logistic_regression,
    n_trials=n_trials_logreg,
    show_progress_bar=True,
)

logreg_results = build_result_table(
    logreg_study,
    "Logistic Regression",
)

display(
    logreg_results[
        select_validation_result_columns(
            logreg_results,
            extra_columns=[
                "Modell",
                "trial_number",
                "objective_value",
                "penalty",
                "alpha",
                "l1_ratio",
                "C",
                "class_weight",
                "max_iter",
                "tol",
                "average",
            ],
        )
    ].head(10)
)

## 6.9. Ergebnisse vergleichen und speichern

Zum Abschluss werden die besten Trials beider Studien zusammengeführt. Für die finale Auswahl werden nicht nur Macro-F1, sondern auch Standardabweichung, Generalization Gap und Laufzeit betrachtet.

In [9]:
optuna_results = pd.concat(
    [
        sgd_hinge_results,
        # logreg_results,
    ],
    ignore_index=True,
)

optuna_results = (
    optuna_results
    .sort_values(
        "objective_value",
        ascending=False,
    )
    .reset_index(drop=True)
)

important_columns = select_validation_result_columns(
    optuna_results,
    extra_columns=[
        "Modell",
        "trial_number",
        "objective_value",
        "penalty",
        "alpha",
        "l1_ratio",
        "C",
        "class_weight",
        "max_iter",
        "tol",
        "average",
    ],
)

display(
    optuna_results[important_columns].head(20)
)

optuna_results_path = (
    optuna_results_dir
    / "optuna_tuning_results.csv"
)

sgd_hinge_results_path = (
    optuna_results_dir
    / "optuna_sgd_hinge_results.csv"
)

logreg_results_path = (
    optuna_results_dir
    / "optuna_logreg_results.csv"
)

optuna_results.to_csv(
    optuna_results_path,
    index=False,
    sep=";",
    encoding="utf-8",
)

sgd_hinge_results.to_csv(
    sgd_hinge_results_path,
    index=False,
    sep=";",
    encoding="utf-8",
)

# logreg_results.to_csv(
#     logreg_results_path,
#     index=False,
#     sep=";",
#     encoding="utf-8",
# )

print("Gespeichert:", optuna_results_path)
print("Gespeichert:", sgd_hinge_results_path)
print("Gespeichert:", logreg_results_path)

,Modell,trial_number,objective_value,penalty,alpha,max_iter,tol,average,l1_ratio,accuracy_mean,...,train_accuracy_mean,train_accuracy_std,train_balanced_accuracy_mean,train_balanced_accuracy_std,train_macro_f1_mean,train_macro_f1_std,train_weighted_f1_mean,train_weighted_f1_std,weighted_f1_mean,weighted_f1_std
0,SGD hinge,16,0.8509,elasticnet,0.000008,2000,0.0001,False,0.066307,0.9410,...,0.9930,0.0003,0.9908,0.0023,0.9693,0.0069,0.9931,0.0003,0.9412,0.0023
1,SGD hinge,42,0.8478,elasticnet,0.000002,5000,0.0001,False,0.115795,0.9389,...,0.9943,0.0005,0.9926,0.0019,0.9740,0.0063,0.9944,0.0005,0.9389,0.0008
2,SGD hinge,41,0.8451,elasticnet,0.000002,5000,0.0001,False,0.076574,0.9375,...,0.9942,0.0006,0.9920,0.0021,0.9759,0.0043,0.9943,0.0006,0.9374,0.0013
3,SGD hinge,33,0.8450,elasticnet,0.000004,1000,0.0001,False,0.087296,0.9387,...,0.9941,0.0005,0.9913,0.0022,0.9731,0.0062,0.9941,0.0005,0.9387,0.0017
4,SGD hinge,22,0.8440,elasticnet,0.000004,2000,0.0001,False,0.107390,0.9395,...,0.9938,0.0007,0.9929,0.0017,0.9711,0.0089,0.9940,0.0007,0.9395,0.0022
5,SGD hinge,30,0.8428,elasticnet,0.000001,1000,0.0001,False,0.090399,0.9361,...,0.9944,0.0006,0.9893,0.0042,0.9712,0.0099,0.9945,0.0005,0.9361,0.0021
6,SGD hinge,39,0.8414,elasticnet,0.000002,5000,0.0001,False,0.074898,0.9378,...,0.9944,0.0010,0.9905,0.0027,0.9712,0.0069,0.9946,0.0009,0.9377,0.0014
7,SGD hinge,11,0.8412,elasticnet,0.000009,2000,0.0001,False,0.297018,0.9391,...,0.9908,0.0003,0.9909,0.0015,0.9686,0.0063,0.9908,0.0003,0.9392,0.0026
8,SGD hinge,43,0.8409,elasticnet,0.000003,5000,0.0001,False,0.110676,0.9371,...,0.9942,0.0004,0.9928,0.0019,0.9726,0.0081,0.9943,0.0004,0.9374,0.0021
9,SGD hinge,24,0.8408,elasticnet,0.000002,2000,0.0001,False,0.119665,0.9370,...,0.9941,0.0012,0.9929,0.0027,0.9710,0.0103,0.9944,0.0008,0.9373,0.0020


Gespeichert: d:\toydev\schwarz-test\data\processed\optuna_tuning_results.csv
Gespeichert: d:\toydev\schwarz-test\data\processed\optuna_sgd_hinge_results.csv
Gespeichert: d:\toydev\schwarz-test\data\processed\optuna_logreg_results.csv


Die Optuna-Ergebnisse zeigen, ob sich die schnellen linearen Modelle durch gezielte Hyperparameterwahl weiter verbessern lassen. Diese Werte stammen aus Cross-Validation bzw. interner Validierung auf den Trainingsdaten und dienen der Modellauswahl. Die endgültige Aussage zur Generalisierung erfolgt erst im separaten Testdatensatz.


## 6.10. Bestes Modell für die finale Evaluation auswählen

Der beste Trial nach Macro-F1 ist ein erster Kandidat für die finale Evaluation. Vor der endgültigen Auswahl sollten jedoch auch `generalization_gap_macro_f1`, `macro_f1_std` und die Laufzeit betrachtet werden. Ein sehr hoher Train-Test-Gap kann auf Overfitting hinweisen.

In [10]:
best_trial_summary = optuna_results.iloc[0]

print("Bestes Modell:", best_trial_summary["Modell"])
print("Trial:", best_trial_summary["trial_number"])
print("Macro-F1:", best_trial_summary["val_macro_f1_mean"])

if "generalization_gap_macro_f1" in best_trial_summary:
    print(
        "Generalization Gap Macro-F1:",
        best_trial_summary["generalization_gap_macro_f1"],
    )

best_trial_display_columns = select_validation_result_columns(
    optuna_results,
    extra_columns=[
        "Modell",
        "trial_number",
        "objective_value",
        "penalty",
        "alpha",
        "l1_ratio",
        "C",
        "class_weight",
        "max_iter",
        "tol",
        "average",
    ],
)

display(
    best_trial_summary[
        [
            column
            for column in best_trial_display_columns
            if column in best_trial_summary.index
        ]
    ].to_frame("Wert")
)

best_params = {
    key: value
    for key, value in best_trial_summary.items()
    if key in [
        "penalty",
        "alpha",
        "l1_ratio",
        "C",
        "class_weight",
        "max_iter",
        "tol",
        "average",
    ]
    and pd.notna(value)
}

if best_trial_summary["Modell"] == "SGD hinge":
    classifier_params = {
        "loss": "hinge",
        "penalty": best_params.get("penalty", "l2"),
        "alpha": float(best_params.get("alpha", 1e-4)),
        "class_weight": "balanced",
        "max_iter": int(best_params.get("max_iter", 1000)),
        "tol": float(best_params.get("tol", 1e-3)),
        "average": bool(best_params.get("average", False)),
        "random_state": 42,
        "n_jobs": -1,
    }

    if classifier_params["penalty"] == "elasticnet":
        classifier_params["l1_ratio"] = float(
            best_params.get("l1_ratio", 0.15)
        )

    final_classifier = SGDClassifier(**classifier_params)

elif best_trial_summary["Modell"] == "Logistic Regression":
    final_classifier = LogisticRegression(
        C=float(best_params.get("C", 1.0)),
        penalty=best_params.get("penalty", "l2"),
        solver="saga",
        class_weight=best_params.get("class_weight", "balanced"),
        max_iter=int(best_params.get("max_iter", 2000)),
        tol=float(best_params.get("tol", 1e-3)),
        n_jobs=-1,
        random_state=42,
    )

else:
    raise ValueError(
        f"Unbekanntes Modell: {best_trial_summary['Modell']}"
    )

best_model_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor_optuna,
        ),
        (
            "classifier",
            final_classifier,
        ),
    ],
    memory=sklearn_memory,
)

best_model_path = save_fitted_model(
    best_model_pipeline,
    f"best_optuna_{best_trial_summary['Modell']}",
    X_train,
    y_train,
)

print("Gespeichertes finales Optuna-Modell:", best_model_path)


Bestes Modell: SGD hinge
Trial: 16
Macro-F1: 0.8509
Generalization Gap Macro-F1: 0.1184


,Wert
Modell,SGD hinge
trial_number,16
objective_value,0.8509
penalty,elasticnet
alpha,0.000008
max_iter,2000
tol,0.0001
average,False
l1_ratio,0.066307
accuracy_mean,0.941


Gespeichertes finales Optuna-Modell: d:\toydev\schwarz-test\models\optuna\best_optuna_sgd_hinge.joblib


# 7. Hyperparameteroptimierung der neuen Modelle

Nach den klassischen scikit-learn-Modellen werden auch die beiden erweiterten NLP-Modelle `fastText` und BERT für eine Hyperparameteroptimierung vorbereitet. Diese Modelle verwenden keine `ColumnTransformer`-Pipeline. Stattdessen werden die verwendeten Spalten zuerst zu einem gemeinsamen Textfeld zusammengeführt.

Die Optimierung erfolgt weiterhin nur auf den Trainingsdaten. Zur Bewertung wird innerhalb der Trainingsdaten ein stratifizierter Validierungssplit verwendet. Der finale Testdatensatz bleibt unverändert unberührt.


### 7.1. Gemeinsame Textbasis für fastText und BERT

fastText und BERT erwarten jeweils einen zusammenhängenden Text pro Beobachtung. Deshalb werden die zuvor ausgewählten Text- und kategorialen Spalten zu einem markierten Text zusammengeführt. Die Feldnamen bleiben erhalten, damit Informationen wie `name`, `geber`, `zweck`, `art` und `jahr` nicht vollständig vermischt werden.


In [6]:
import time
import unicodedata

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


nlp_model_output_dir = model_output_dir / "new_nlp_models"
nlp_model_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)


nlp_feature_columns = feature_columns_optuna.copy()

X_train_nlp = X_train.copy()
X_train_nlp["model_text"] = combine_columns_as_text(
    X_train_nlp,
    nlp_feature_columns,
)

nlp_train_texts, nlp_valid_texts, nlp_y_train, nlp_y_valid = train_test_split(
    X_train_nlp["model_text"],
    y_train,
    test_size=0.1,
    random_state=42,
    stratify=y_train,
)

print("Verwendete Spalten für neue NLP-Modelle:")
display(
    pd.DataFrame(
        {"Trainingsspalte": nlp_feature_columns}
    )
)

print("Trainingssplit für BERT-Holdout:", len(nlp_y_train))
print("Validierungssplit für BERT-Holdout:", len(nlp_y_valid))

display(
    X_train_nlp[["model_text"]].head(3)
)


Verwendete Spalten für neue NLP-Modelle:


,Trainingsspalte
0,name_standardised
1,geber_standardised
2,anschrift_standardised
3,zweck_standardised
4,art_standardised
5,jahr


Trainingssplit: 41328
Validierungssplit: 4592


,model_text
id,
33655,name: cashmere radio e. v. geber: senatsverwal...
105910,name: merantix labs gmbh geber: senatsverwaltu...
67465,name: georg kolbe-stiftung geber: senatsverwal...


### 7.2. fastText mit Optuna optimieren

fastText ist schnell genug, um mehrere Hyperparameterkombinationen auszuprobieren. Optimiert werden unter anderem Lernrate, Anzahl der Epochen, Wort-n-Gramme, Zeichen-n-Gramme und die Einbettungsdimension. Als Zielwert wird wie zuvor der Macro-F1-Score auf dem internen Validierungssplit verwendet.


In [12]:
fasttext_output_dir = nlp_model_output_dir / "fasttext"
fasttext_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)


fasttext_label_mapping = (
    pd.DataFrame(
        {"politikbereich": sorted(y_train.unique())}
    )
    .assign(
        fasttext_label=lambda dataframe: dataframe["politikbereich"].map(
            make_fasttext_label
        )
    )
)

if fasttext_label_mapping["fasttext_label"].duplicated().any():
    raise ValueError(
        "Nicht eindeutige fastText-Labels gefunden. "
        "Bitte Label-Erzeugung prüfen."
    )

fasttext_label_to_class = dict(
    zip(
        fasttext_label_mapping["fasttext_label"],
        fasttext_label_mapping["politikbereich"],
    )
)

fasttext_label_mapping.to_csv(
    fasttext_output_dir / "fasttext_label_mapping.csv",
    index=False,
    sep=";",
    encoding="utf-8",
)


def write_fasttext_file(
    path,
    texts,
    labels,
):
    lines = (
        labels.map(make_fasttext_label)
        + " "
        + texts
        .fillna("")
        .astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace("\r", " ", regex=False)
    )

    path.write_text(
        "\n".join(lines.tolist()),
        encoding="utf-8",
    )


fasttext_full_train_path = fasttext_output_dir / "full_train.txt"

write_fasttext_file(
    fasttext_full_train_path,
    X_train_nlp["model_text"],
    y_train,
)

try:
    import fasttext

    def evaluate_fasttext_model(
        model,
        texts,
        y_true,
    ):
        predicted_labels, _ = model.predict(
            texts.tolist(),
            k=1,
        )

        y_pred = pd.Series(
            [
                fasttext_label_to_class.get(label_group[0], label_group[0])
                if len(label_group) > 0
                else None
                for label_group in predicted_labels
            ],
            index=y_true.index,
        )

        return {
            "accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            ),
            "weighted_f1": f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0,
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y_true,
                y_pred,
            ),
        }


    def objective_fasttext(trial):
        params = {
            "epoch": trial.suggest_int("epoch", 5, 40),
            "lr": trial.suggest_float("lr", 0.05, 1.0, log=True),
            "wordNgrams": trial.suggest_int("wordNgrams", 1, 3),
            "dim": trial.suggest_categorical("dim", [50, 100, 200]),
            "minn": trial.suggest_categorical("minn", [0, 2, 3]),
            "loss": trial.suggest_categorical("loss", ["softmax", "ova"]),
        }

        if params["minn"] == 0:
            params["maxn"] = 0
        else:
            params["maxn"] = trial.suggest_int("maxn", 4, 7)

        fold_metric_rows = []
        total_training_time = 0.0

        for fold_number, (train_index, valid_index) in enumerate(
            cv.split(X_train_nlp["model_text"], y_train),
            start=1,
        ):
            fold_train_texts = X_train_nlp["model_text"].iloc[train_index]
            fold_valid_texts = X_train_nlp["model_text"].iloc[valid_index]
            fold_y_train = y_train.iloc[train_index]
            fold_y_valid = y_train.iloc[valid_index]

            fold_train_path = (
                fasttext_output_dir
                / f"trial_{trial.number}_fold_{fold_number}_train.txt"
            )
            write_fasttext_file(
                fold_train_path,
                fold_train_texts,
                fold_y_train,
            )

            start_time = time.perf_counter()
            model = fasttext.train_supervised(
                input=str(fold_train_path),
                verbose=0,
                **params,
            )
            training_time = time.perf_counter() - start_time
            total_training_time += training_time

            metrics = evaluate_fasttext_model(
                model,
                fold_valid_texts,
                fold_y_valid,
            )
            metrics["fold"] = fold_number
            metrics["fit_time_seconds"] = training_time
            fold_metric_rows.append(metrics)

        fold_metrics = pd.DataFrame(fold_metric_rows)

        summary = {
            "CV_Folds": n_splits,
            "validation_strategy": "shared_stratified_4_fold_cv",
            "fit_time_total_seconds": round(total_training_time, 2),
        }

        for metric_name in [
            "accuracy",
            "macro_f1",
            "weighted_f1",
            "balanced_accuracy",
        ]:
            summary[f"val_{metric_name}_mean"] = round(
                fold_metrics[metric_name].mean(),
                4,
            )
            summary[f"val_{metric_name}_std"] = round(
                fold_metrics[metric_name].std(ddof=0),
                4,
            )
            summary[f"{metric_name}_mean"] = summary[
                f"val_{metric_name}_mean"
            ]
            summary[f"{metric_name}_std"] = summary[
                f"val_{metric_name}_std"
            ]

        for metric_name, metric_value in summary.items():
            trial.set_user_attr(metric_name, metric_value)

        with mlflow.start_run(
            run_name=f"fastText trial {trial.number}",
            nested=True,
        ):
            mlflow.log_params(params)
            mlflow.log_param("model", "fastText")
            mlflow.log_param(
                "validation_strategy",
                "shared_stratified_4_fold_cv",
            )
            for metric_name, metric_value in summary.items():
                if isinstance(metric_value, (int, float)):
                    mlflow.log_metric(metric_name, metric_value)

        return summary[f"val_{main_metric}_mean"]


    fasttext_study = optuna.create_study(
        study_name="fasttext_text_categorical_macro_f1_cv4",
        direction="maximize",
        storage=optuna_storage_url,
        load_if_exists=True,
        sampler=sampler,
    )

    n_trials_fasttext = 25

    fasttext_study.optimize(
        objective_fasttext,
        n_trials=n_trials_fasttext,
        show_progress_bar=True,
    )

    fasttext_tuning_results = build_result_table(
        fasttext_study,
        "fastText",
    )

    fasttext_tuning_results_path = (
        optuna_results_dir
        / "optuna_fasttext_results.csv"
    )
    fasttext_tuning_results.to_csv(
        fasttext_tuning_results_path,
        index=False,
        sep=";",
        encoding="utf-8",
    )

    best_fasttext_params = fasttext_study.best_trial.params.copy()
    if best_fasttext_params.get("minn", 0) == 0:
        best_fasttext_params["maxn"] = 0

    best_fasttext_model = fasttext.train_supervised(
        input=str(fasttext_full_train_path),
        verbose=0,
        **best_fasttext_params,
    )

    best_fasttext_model_path = (
        fasttext_output_dir
        / "best_fasttext_politikbereich.bin"
    )
    best_fasttext_model.save_model(str(best_fasttext_model_path))

    print("Bestes fastText-Modell gespeichert:", best_fasttext_model_path)
    display(fasttext_tuning_results.head(10))

except ImportError:
    fasttext_tuning_results = pd.DataFrame()
    print(
        "fastText ist nicht installiert. "
        "Installiere bei Bedarf fasttext und führe diese Zelle erneut aus."
    )


[I 2026-07-27 21:39:02,793] A new study created in RDB with name: fasttext_text_categorical_macro_f1
Best trial: 0. Best value: 0.736124:   4%|▍         | 1/25 [00:36<14:30, 36.29s/it]

[I 2026-07-27 21:39:39,069] Trial 0 finished with value: 0.7361235503014153 and parameters: {'epoch': 18, 'lr': 0.8627358286640178, 'wordNgrams': 3, 'dim': 50, 'minn': 2, 'loss': 'softmax', 'maxn': 7}. Best is trial 0 with value: 0.7361235503014153.


Best trial: 0. Best value: 0.736124:   8%|▊         | 2/25 [02:15<28:07, 73.37s/it]

[I 2026-07-27 21:41:18,408] Trial 1 finished with value: 0.5114159838543018 and parameters: {'epoch': 34, 'lr': 0.09445600138094695, 'wordNgrams': 1, 'dim': 200, 'minn': 3, 'loss': 'ova', 'maxn': 5}. Best is trial 0 with value: 0.7361235503014153.


Best trial: 2. Best value: 0.811075:  12%|█▏        | 3/25 [02:27<16:33, 45.14s/it]

...

Best trial: 19. Best value: 0.818185:  96%|█████████▌| 24/25 [15:25<00:52, 52.05s/it]

[I 2026-07-27 21:54:28,690] Trial 23 finished with value: 0.8176070033696257 and parameters: {'epoch': 36, 'lr': 0.6226184609656309, 'wordNgrams': 3, 'dim': 200, 'minn': 0, 'loss': 'softmax'}. Best is trial 19 with value: 0.8181848946121342.


Best trial: 19. Best value: 0.818185: 100%|██████████| 25/25 [15:51<00:00, 38.07s/it]


[I 2026-07-27 21:54:54,569] Trial 24 finished with value: 0.8151256890492379 and parameters: {'epoch': 31, 'lr': 0.33269105289670226, 'wordNgrams': 2, 'dim': 200, 'minn': 0, 'loss': 'softmax'}. Best is trial 19 with value: 0.8181848946121342.
Bestes fastText-Modell gespeichert: d:\toydev\schwarz-test\models\optuna\new_nlp_models\fasttext\best_fasttext_politikbereich.bin


,Modell,trial_number,objective_value,epoch,lr,wordNgrams,dim,minn,loss,maxn,accuracy,balanced_accuracy,macro_f1,training_time_seconds,weighted_f1
0,fastText,19,0.818185,37,0.635238,3,200,0,softmax,NaN,0.938153,0.804543,0.818185,32.64,0.937909
1,fastText,23,0.817607,36,0.622618,3,200,0,softmax,NaN,0.938807,0.800787,0.817607,32.78,0.938422
2,fastText,15,0.817482,31,0.722380,2,100,0,softmax,NaN,0.939024,0.803813,0.817482,18.78,0.938782
3,fastText,21,0.816971,37,0.641852,3,200,0,softmax,NaN,0.937500,0.800961,0.816971,32.28,0.937075
4,fastText,18,0.815667,38,0.655307,3,100,0,softmax,NaN,0.937500,0.798312,0.815667,24.13,0.937055
5,fastText,16,0.815214,31,0.772386,3,100,0,softmax,NaN,0.937064,0.798791,0.815214,20.79,0.936952
6,fastText,24,0.815126,31,0.332691,2,200,0,softmax,NaN,0.937936,0.801873,0.815126,25.20,0.937623
7,fastText,13,0.815082,29,0.819331,2,100,0,softmax,NaN,0.936847,0.800705,0.815082,18.34,0.936628
8,fastText,11,0.814713,27,0.468854,2,100,0,softmax,NaN,0.937718,0.801136,0.814713,15.92,0.937305
9,fastText,12,0.813116,28,0.453600,2,100,0,softmax,NaN,0.936847,0.799917,0.813116,16.89,0.936587


### 7.3. BERT mit Optuna vorbereiten

BERT kann ebenfalls optimiert werden, ist aber deutlich rechenintensiver. Deshalb ist die BERT-Optimierung standardmäßig deaktiviert. In Colab mit GPU kann `run_bert_tuning = True` gesetzt werden. Optimiert werden nur wenige zentrale Parameter: Lernrate, Batch Size, Weight Decay, maximale Sequenzlänge und Anzahl der Epochen.


In [ ]:
bert_output_dir = nlp_model_output_dir / "bert"
bert_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

run_bert_tuning = False
n_trials_bert = 5
bert_model_checkpoint = "deepset/gbert-base"

bert_label_encoder = LabelEncoder()
bert_label_encoder.fit(y_train)

bert_label_mapping = pd.DataFrame(
    {
        "label_id": range(len(bert_label_encoder.classes_)),
        "politikbereich": bert_label_encoder.classes_,
    }
)
bert_label_mapping.to_csv(
    bert_output_dir / "bert_label_mapping.csv",
    index=False,
    sep=";",
    encoding="utf-8",
)
dump(
    bert_label_encoder,
    bert_output_dir / "bert_label_encoder.joblib",
)

bert_y_train = bert_label_encoder.transform(nlp_y_train)
bert_y_valid = bert_label_encoder.transform(nlp_y_valid)


if run_bert_tuning:
    try:
        import shutil
        import torch
        from transformers import (
            BertForSequenceClassification,
            BertTokenizer,
            Trainer,
            TrainingArguments,
        )

        class EncodedPolitikbereichDataset(torch.utils.data.Dataset):
            """Vor-tokenisierter Datensatz für BERT."""

            def __init__(
                self,
                encodings,
                labels,
            ):
                self.encodings = encodings
                self.labels = labels

            def __len__(self):
                return len(self.labels)

            def __getitem__(self, index):
                item = {
                    key: torch.tensor(value[index])
                    for key, value in self.encodings.items()
                }
                item["labels"] = torch.tensor(
                    self.labels[index],
                    dtype=torch.long,
                )
                return item

        def make_training_arguments(
            output_dir,
            max_epochs,
            learning_rate,
            batch_size,
            weight_decay,
        ):
            common_kwargs = {
                "output_dir": str(output_dir),
                "num_train_epochs": max_epochs,
                "learning_rate": learning_rate,
                "per_device_train_batch_size": batch_size,
                "per_device_eval_batch_size": max(batch_size, 8),
                "weight_decay": weight_decay,
                "logging_steps": 100,
                "save_strategy": "epoch",
                "load_best_model_at_end": True,
                "metric_for_best_model": "macro_f1",
                "greater_is_better": True,
                "report_to": "none",
            }

            try:
                return TrainingArguments(
                    eval_strategy="epoch",
                    **common_kwargs,
                )
            except TypeError:
                return TrainingArguments(
                    evaluation_strategy="epoch",
                    **common_kwargs,
                )

        def compute_bert_metrics(eval_prediction):
            logits, labels = eval_prediction
            predictions = np.argmax(logits, axis=-1)

            return {
                "val_accuracy": accuracy_score(labels, predictions),
                "val_macro_f1": f1_score(
                    labels,
                    predictions,
                    average="macro",
                    zero_division=0,
                ),
                "val_weighted_f1": f1_score(
                    labels,
                    predictions,
                    average="weighted",
                    zero_division=0,
                ),
                "val_balanced_accuracy": balanced_accuracy_score(
                    labels,
                    predictions,
                ),
            }

        bert_tokenizer = BertTokenizer.from_pretrained(
            bert_model_checkpoint,
            token=hf_token,
        )

        def objective_bert(trial):
            learning_rate = trial.suggest_float(
                "learning_rate",
                1e-5,
                5e-5,
                log=True,
            )
            batch_size = trial.suggest_categorical(
                "batch_size",
                [8, 16],
            )
            weight_decay = trial.suggest_float(
                "weight_decay",
                0.0,
                0.1,
            )
            max_length = trial.suggest_categorical(
                "max_length",
                [128, 256],
            )
            num_train_epochs = trial.suggest_categorical(
                "num_train_epochs",
                [1, 2, 3],
            )

            train_encodings = bert_tokenizer(
                nlp_train_texts.tolist(),
                truncation=True,
                padding="max_length",
                max_length=max_length,
            )
            valid_encodings = bert_tokenizer(
                nlp_valid_texts.tolist(),
                truncation=True,
                padding="max_length",
                max_length=max_length,
            )

            train_dataset = EncodedPolitikbereichDataset(
                train_encodings,
                bert_y_train,
            )
            valid_dataset = EncodedPolitikbereichDataset(
                valid_encodings,
                bert_y_valid,
            )

            model = BertForSequenceClassification.from_pretrained(
                bert_model_checkpoint,
                token=hf_token,
                num_labels=len(bert_label_encoder.classes_),
                id2label={
                    index: label
                    for index, label in enumerate(bert_label_encoder.classes_)
                },
                label2id={
                    label: index
                    for index, label in enumerate(bert_label_encoder.classes_)
                },
            )

            trial_output_dir = bert_output_dir / f"trial_{trial.number}"
            training_args = make_training_arguments(
                trial_output_dir,
                num_train_epochs,
                learning_rate,
                batch_size,
                weight_decay,
            )

            trainer = Trainer(
                model=model,
                args=training_args,
                train_dataset=train_dataset,
                eval_dataset=valid_dataset,
                compute_metrics=compute_bert_metrics,
            )

            start_time = time.perf_counter()
            trainer.train()
            eval_metrics = trainer.evaluate()
            training_time = time.perf_counter() - start_time

            metrics = {
                key.replace("eval_", ""): value
                for key, value in eval_metrics.items()
                if key.startswith("eval_")
            }
            metrics["training_time_seconds"] = round(training_time, 2)

            for metric_name, metric_value in metrics.items():
                trial.set_user_attr(metric_name, metric_value)

            with mlflow.start_run(
                run_name=f"BERT trial {trial.number}",
                nested=True,
            ):
                mlflow.log_param("model", bert_model_checkpoint)
                mlflow.log_param("learning_rate", learning_rate)
                mlflow.log_param("batch_size", batch_size)
                mlflow.log_param("weight_decay", weight_decay)
                mlflow.log_param("max_length", max_length)
                mlflow.log_param("num_train_epochs", num_train_epochs)
                for metric_name, metric_value in metrics.items():
                    if isinstance(metric_value, (int, float)):
                        mlflow.log_metric(metric_name, metric_value)

            shutil.rmtree(
                trial_output_dir,
                ignore_errors=True,
            )

            return metrics[main_metric]

        bert_study = optuna.create_study(
            study_name="bert_text_categorical_macro_f1",
            direction="maximize",
            storage=optuna_storage_url,
            load_if_exists=True,
            sampler=sampler,
        )

        bert_study.optimize(
            objective_bert,
            n_trials=n_trials_bert,
            show_progress_bar=True,
        )

        bert_tuning_results = build_result_table(
            bert_study,
            "German BERT",
        )

        bert_tuning_results_path = (
            optuna_results_dir
            / "optuna_bert_results.csv"
        )
        bert_tuning_results.to_csv(
            bert_tuning_results_path,
            index=False,
            sep=";",
            encoding="utf-8",
        )

        display(bert_tuning_results.head(10))

    except ImportError:
        bert_tuning_results = pd.DataFrame()
        print(
            "transformers und/oder torch sind nicht installiert. "
            "Installiere die Pakete und führe diese Zelle erneut aus."
        )
else:
    bert_tuning_results = pd.DataFrame()
    print(
        "BERT-Tuning ist deaktiviert. "
        "Setze run_bert_tuning = True, um es in einer GPU-Umgebung auszuführen."
    )


### 7.4. Ergebnisse der neuen Modelle zusammenführen

Die Ergebnisse der neuen NLP-Modelle werden separat gespeichert. Dadurch bleiben die klassischen Optuna-Ergebnisse reproduzierbar, während fastText und BERT als erweiterte Vergleichsmodelle ausgewertet werden können.


In [13]:
extended_result_frames = []

if "fasttext_tuning_results" in globals() and not fasttext_tuning_results.empty:
    extended_result_frames.append(fasttext_tuning_results)

# if "bert_tuning_results" in globals() and not bert_tuning_results.empty:
#     extended_result_frames.append(bert_tuning_results)

if extended_result_frames:
    extended_nlp_tuning_results = (
        pd.concat(
            extended_result_frames,
            ignore_index=True,
        )
        .sort_values(
            "objective_value",
            ascending=False,
        )
        .reset_index(drop=True)
    )
else:
    extended_nlp_tuning_results = pd.DataFrame()

extended_nlp_results_path = (
    optuna_results_dir
    / "optuna_extended_nlp_results.csv"
)

extended_nlp_tuning_results.to_csv(
    extended_nlp_results_path,
    index=False,
    sep=";",
    encoding="utf-8",
)

print("Gespeichert:", extended_nlp_results_path)

if not extended_nlp_tuning_results.empty:
    extended_nlp_display_columns = select_validation_result_columns(
        extended_nlp_tuning_results,
        extra_columns=[
            "Modell",
            "trial_number",
            "objective_value",
            "epoch",
            "lr",
            "wordNgrams",
            "dim",
            "minn",
            "maxn",
            "loss",
        ],
    )
    display(
        extended_nlp_tuning_results[
            extended_nlp_display_columns
        ].head(20)
    )
else:
    display(extended_nlp_tuning_results)


Gespeichert: d:\toydev\schwarz-test\data\processed\optuna_extended_nlp_results.csv


,Modell,trial_number,objective_value,epoch,lr,wordNgrams,dim,minn,loss,maxn,accuracy,balanced_accuracy,macro_f1,training_time_seconds,weighted_f1
0,fastText,19,0.818185,37,0.635238,3,200,0,softmax,NaN,0.938153,0.804543,0.818185,32.64,0.937909
1,fastText,23,0.817607,36,0.622618,3,200,0,softmax,NaN,0.938807,0.800787,0.817607,32.78,0.938422
2,fastText,15,0.817482,31,0.722380,2,100,0,softmax,NaN,0.939024,0.803813,0.817482,18.78,0.938782
3,fastText,21,0.816971,37,0.641852,3,200,0,softmax,NaN,0.937500,0.800961,0.816971,32.28,0.937075
4,fastText,18,0.815667,38,0.655307,3,100,0,softmax,NaN,0.937500,0.798312,0.815667,24.13,0.937055
5,fastText,16,0.815214,31,0.772386,3,100,0,softmax,NaN,0.937064,0.798791,0.815214,20.79,0.936952
6,fastText,24,0.815126,31,0.332691,2,200,0,softmax,NaN,0.937936,0.801873,0.815126,25.20,0.937623
7,fastText,13,0.815082,29,0.819331,2,100,0,softmax,NaN,0.936847,0.800705,0.815082,18.34,0.936628
8,fastText,11,0.814713,27,0.468854,2,100,0,softmax,NaN,0.937718,0.801136,0.814713,15.92,0.937305
9,fastText,12,0.813116,28,0.453600,2,100,0,softmax,NaN,0.936847,0.799917,0.813116,16.89,0.936587


Die erweiterten NLP-Ergebnisse werden separat zusammengeführt, weil fastText und BERT nicht dieselbe scikit-learn-Pipeline verwenden. fastText kann mit begrenzter Rechenzeit relativ schnell getestet werden. BERT bleibt dagegen optional, da ein systematisches Fine-Tuning deutlich höhere GPU-Ressourcen erfordert.
